<a href="https://colab.research.google.com/github/nebjj/RAG/blob/main/RAG_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pypdf \
    transformers accelerate

In [ ]:
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))
print("Uploaded:", file_name)

Saving Animal Kingdom Revision Notes_Class 11 Biology Chapter 4.pdf to Animal Kingdom Revision Notes_Class 11 Biology Chapter 4 (1).pdf
Uploaded: Animal Kingdom Revision Notes_Class 11 Biology Chapter 4 (1).pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_name)
documents = loader.load()

print("Pages:", len(documents))

/tmp/ipykernel_1086/3181399108.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages: 34


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 58


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_chroma import Chroma

databasev = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_collection"
)

In [ ]:
question = input("Ask a question about the document: ")

Ask a question about the document: What is Pseudocoelomates?


In [ ]:
results = databasev.similarity_search(
    question,
    k=4
)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n Retrieved Chunk {i+1}")
    print(doc.page_content)


 Retrieved Chunk 1
Eg: Platyhelminthes 
• Pseudocoelomates: Animals having false coelom.  Body cavity is not formed from the 
mesodermal epithelium . Mesoderm appears  as scattered pouches in between ectoderm and 
endoderm. In these animals pseudocoelom is formed from the embryonic cavity called blastocoel. 
                      Eg: Aschelminthes 
• Coelomates or Eucoelomates: Animals with true body cavity 
                     Eg: Annelids, Molluscs, Arthropods, Echinoderms, Hemichordates, and    Chordates. 
 
 
    4.1.5 SEGMENTATION 
In some animals the entire length of body is transversely divided in to a number of ring like parts called 
segments. This phenomenon of segmentation is called metamerism.  
This pattern is clearly seen in annelids like earthworm and arthropods like millipedes and centipedes. 
In earthworms the segmentation helps in locomotion. 
In arthropods the met americ segments may be paired appe ndages for various functions such as legs for

 Retrieved Chunk 2
E

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in results
)

In [ ]:
context

'Eg: Platyhelminthes \n• Pseudocoelomates: Animals having false coelom.  Body cavity is not formed from the \nmesodermal epithelium . Mesoderm appears  as scattered pouches in between ectoderm and \nendoderm. In these animals pseudocoelom is formed from the embryonic cavity called blastocoel. \n                      Eg: Aschelminthes \n• Coelomates or Eucoelomates: Animals with true body cavity \n                     Eg: Annelids, Molluscs, Arthropods, Echinoderms, Hemichordates, and    Chordates. \n \n \n    4.1.5 SEGMENTATION \nIn some animals the entire length of body is transversely divided in to a number of ring like parts called \nsegments. This phenomenon of segmentation is called metamerism.  \nThis pattern is clearly seen in annelids like earthworm and arthropods like millipedes and centipedes. \nIn earthworms the segmentation helps in locomotion. \nIn arthropods the met americ segments may be paired appe ndages for various functions such as legs for\n\nEg: Platyhelminthes \n•

In [ ]:
prompt = f"""
You are a helpful document question-answering assistant.

Answer the user's question using ONLY the information
provided in the context.

If the answer cannot be found in the context,
say "I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is the capital of India?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(response)

The capital of India is New Delhi.


In [ ]:
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=4096
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("\nAnswer:")
print(answer)


Answer:
Pseudocoelomates are animals that have false coelom. Their body cavity is not formed from the mesodermal epithelium but rather from scattered pouches in between the ectoderm and endoderm. They are characterized by pseudocoelom being formed from the embryonic cavity called blastocoel.


In [ ]:
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break


    results = vectorstore.similarity_search(
        question,
        k=3
    )


    context = "\n\n".join(
        doc.page_content
        for doc in results
    )


    prompt = f"""
Answer the question using ONLY the provided context.

If the answer cannot be found in the context,
say "I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    print("\nAnswer:")
    print(answer)


Ask a question (type 'exit' to stop): EXIT


In [ ]:
!pip install -q streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [ ]:
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /content/cloudflared.log 2>&1 &

In [ ]:
!sleep 5
!cat /content/cloudflared.log

In [ ]:
import streamlit as st
import torch

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="RAG Chatbot",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 RAG Document Chatbot")
st.write("Upload a PDF and ask questions about it.")


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

@st.cache_resource
def load_embeddings():

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embeddings


# ============================================================
# LOAD LLM
# ============================================================

@st.cache_resource
def load_model():

    model_name = "Qwen/Qwen2.5-0.5B-Instruct"

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto"
    )

    return tokenizer, model


# ============================================================
# CREATE VECTOR DATABASE
# ============================================================

@st.cache_resource
def create_vectorstore(file_path):

    # Load PDF
    loader = PyPDFLoader(file_path)

    documents = loader.load()

    # Split documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(documents)

    # Embeddings
    embeddings = load_embeddings()

    # Vector database
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="rag_collection"
    )

    return vectorstore, len(documents), len(chunks)


# ============================================================
# GENERATE ANSWER
# ============================================================

def generate_answer(question, vectorstore, tokenizer, model):

    # --------------------------------------------------------
    # 1. RETRIEVAL
    # --------------------------------------------------------

    results = vectorstore.similarity_search(
        question,
        k=3
    )

    # Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content
        for doc in results
    )

    # --------------------------------------------------------
    # 2. PROMPT
    # --------------------------------------------------------

    prompt = f"""
Answer the question using ONLY the information
provided in the context.

If the answer cannot be found in the context,
say:

"I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""

    # --------------------------------------------------------
    # 3. LLM GENERATION
    # --------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer, results


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header("Document")

uploaded_file = st.sidebar.file_uploader(
    "Upload a PDF",
    type=["pdf"]
)


# ============================================================
# INITIALIZE CHAT HISTORY
# ============================================================

if "messages" not in st.session_state:

    st.session_state.messages = []


# ============================================================
# PROCESS PDF
# ============================================================

if uploaded_file is not None:

    # Save uploaded PDF temporarily
    with open("uploaded_document.pdf", "wb") as f:

        f.write(
            uploaded_file.getbuffer()
        )

    # Create vector database
    with st.spinner("Processing document..."):

        vectorstore, pages, chunks = create_vectorstore(
            "uploaded_document.pdf"
        )

    st.sidebar.success("PDF processed successfully!")

    st.sidebar.write(
        f"📄 Pages: {pages}"
    )

    st.sidebar.write(
        f"🧩 Chunks: {chunks}"
    )

    # Load models
    with st.spinner("Loading AI model..."):

        tokenizer, model = load_model()


    # ========================================================
    # DISPLAY PREVIOUS CHAT MESSAGES
    # ========================================================

    for message in st.session_state.messages:

        with st.chat_message(message["role"]):

            st.markdown(
                message["content"]
            )


    # ========================================================
    # CHAT INPUT
    # ========================================================

    question = st.chat_input(
        "Ask something about the document.."
    )


    if question:

        # ----------------------------------------------------
        # DISPLAY USER QUESTION
        # ----------------------------------------------------

        with st.chat_message("user"):

            st.markdown(question)


        # Save user message
        st.session_state.messages.append(
            {
                "role": "user",
                "content": question
            }
        )


        # ----------------------------------------------------
        # GENERATE RESPONSE
        # ----------------------------------------------------

        with st.chat_message("assistant"):

            with st.spinner("Thinking..."):

                answer, retrieved_docs = generate_answer(
                    question,
                    vectorstore,
                    tokenizer,
                    model
                )

            st.markdown(answer)


        # Save assistant response
        st.session_state.messages.append(
            {
                "role": "assistant",
                "content": answer
            }
        )


        # ----------------------------------------------------
        # SHOW RETRIEVED CHUNKS
        # ----------------------------------------------------

        with st.expander(
            "View Retrieved Context"
        ):

            for i, doc in enumerate(
                retrieved_docs
            ):

                st.markdown(
                    f"### Retrieved Chunk {i + 1}"
                )

                st.write(
                    doc.page_content
                )

else:

    st.info(
        "Upload a PDF from the sidebar to start chatting."
    )